# 03 · Referees

**Question:** Is the drop in the card gap about referees, and did some referees change more than others?

Only the Premier League files name the referee, so section 2 uses the Premier League only.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd

from ghost_games import figures
from ghost_games.pipeline import build_matches
from ghost_games.referees import booking_rate_gap, referee_table

matches = build_matches("..")
main = matches[matches.crowd_status.isin(["fans", "empty"])]

## 1. Who gets booked: home or away players?
Yellow cards per match, then cards **per foul**. Fouls mostly reflect how teams play; cards per foul mostly reflect how strictly the referee punishes.

In [ ]:
cards = main.groupby("closed_doors")[
    ["home_yellows", "away_yellows", "home_fouls", "away_fouls"]
].mean()
cards.index = cards.index.map({0: "with fans", 1: "empty"})
cards.round(2)

In [ ]:
booking = pd.Series(
    {
        "with fans": booking_rate_gap(main[main.closed_doors == 0]),
        "empty": booking_rate_gap(main[main.closed_doors == 1]),
    },
    name="away minus home cards per foul",
)
booking.round(4)

## 2. Did some referees change more than others? (Premier League)

In [ ]:
table = referee_table(matches[matches.league == "E0"])
Path("../results").mkdir(exist_ok=True)
table.to_csv("../results/referees.csv")
table

In [ ]:
fig = figures.plot_referees(table)
fig.savefig("../figures/referees.png", dpi=200, facecolor=figures.SURFACE)
# Takeaway: TODO

**Reading the table:** `se` is the uncertainty of each referee's change. `trust_weight` near 0 means a referee's own number is mostly noise, so `shrunk_change` uses the overall change instead.